# Notebook RAG Retriever dense + FAISS (Qwen 0.6B), LLM Qwen 7B vs 14B



## 1) Imports

Cette cellule charge toutes les dépendances nécessaires :
- manipulation de données (pandas, datasets),
- embeddings + indexation/recherche (FAISS),
- génération LLM (transformers),
- métriques SQuAD (evaluate).

In [1]:
import os
import json
import ast
import re
import gc
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from tqdm import tqdm

import torch
import torch.nn.functional as F

import faiss 
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

/home/rayane/Desktop/Paris Cité/M1/PPD/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Paramètres (LLM, retriever, chemins, tailles)

Cette cellule définit les paramètres principaux.

- **Entrées :** aucune.
- **Ce que fait la cellule :**
  - choisit le LLM via `LLM_NAME, LLM_TAG` (7B ou 14B),
  - définit le retriever dense `RETRIEVER_NAME`,
  - fixe les chemins `train.csv` et `validation.csv`,
  - fixe la graine `SEED`,
  - configure le retrieval :
    - `TOP_K`, `CHUNK_WORDS`, `CHUNK_OVERLAP`,
    - `RET_MAX_TOKENS`, `RET_BATCH_SIZE`, `QUERY_BATCH_SIZE`,
  - configure la génération :
    - `MAX_INPUT_TOKENS`, `MAX_NEW_TOKENS`, `GEN_BATCH_SIZE`,
  - définit le cache FAISS (index + chunks) et les fichiers de sortie.
- **Sorties :** aucune (variables globales).

In [2]:
# LLM (generator)
LLM_NAME, LLM_TAG = "Qwen/Qwen2.5-7B-Instruct", "7B" #7B
#LLM_NAME, LLM_TAG = "JungZoona/T3Q-qwen2.5-14b-v1.0-e3", "14B" #14B


# Dense retriever (embeddings)
RETRIEVER_NAME = "Qwen/Qwen3-Embedding-0.6B"

DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"

SEED = 42

# Retrieval / indexing
TOP_K = 5
CHUNK_WORDS = 200
CHUNK_OVERLAP = 50
RET_MAX_TOKENS = 512
RET_BATCH_SIZE = 64
QUERY_BATCH_SIZE = 64

# Generation
MAX_INPUT_TOKENS = 768
MAX_NEW_TOKENS = 32
GEN_BATCH_SIZE = 1  



# Cache 
CACHE_DIR = "results/rag_cache"
INDEX_PATH = os.path.join(
    CACHE_DIR,
    f"faiss_ip_dim1024_val_cw{CHUNK_WORDS}_ov{CHUNK_OVERLAP}.index"
)
CHUNKS_PATH = os.path.join(
    CACHE_DIR,
    f"chunks_val_cw{CHUNK_WORDS}_ov{CHUNK_OVERLAP}.jsonl"
)

# Outputs
METRICS_CSV = f"results/metrics_table_qwen{LLM_TAG}_rag_validation.csv"
PRED200_CSV = f"results/manual_eval_qwen{LLM_TAG}_rag_validation.csv"

DEVICE = "cuda" 

## 3) Nettoyage `answers` + normalisation dataset + prompt + post-traitement

Ces fonctions mettent le dataset au format attendu (SQuAD) et préparent les entrées/sorties du LLM.

### `parse_answers_field(x)`
- **Entrée :** `x` (texte) = contenu brut de la colonne `answers`.
- **Ce que fait la fonction :**
  - supprime la forme `array(..., dtype=...)`,
  - transforme en dictionnaire Python,
  - force les champs en listes (`text`, `answer_start`).
- **Sortie :** dict `{"text": [...], "answer_start": [...]}`.

### `normalize_dataset(example)`
- **Entrée :** une ligne du dataset (dict).
- **Ce que fait la fonction :**
  - applique `parse_answers_field` sur `answers`,
  - convertit `id` en string.
- **Sortie :** la ligne corrigée.

### `build_prompt(question, tokenizer, context=None)`
- **Entrées :**
  - `question` : question,
  - `tokenizer` : tokenizer du LLM,
  - `context` : contexte récupéré (RAG) ou `None`.
- **Ce que fait la fonction :**
  - construit un prompt chat “Context + Question → Answer”,
  - utilise le chat template si disponible.
- **Sortie :** prompt texte.

### `clean_generation(text)`
- **Entrée :** texte généré.
- **Ce que fait la fonction :**
  - garde la première ligne,
  - supprime des préfixes (`Answer:`, etc.).
- **Sortie :** réponse finale nettoyée.

In [ ]:
def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans

def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example

def build_prompt(question, tokenizer, context = None):
    question = str(question).strip()

    system_msg = (
        "You are a helpful assistant. "
        "Answer the question with a short phrase. "
        "Do not add explanations. "
        "If the context does not contain the answer, say: I don't know."
    )

    if context is not None and len(context.strip()) > 0:
        user_msg = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    else:
        user_msg = f"Question: {question}\nAnswer:"

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return f"{system_msg}\n\n{user_msg}"


def clean_generation(text):
    t = (text or "").strip()
    t = t.split("\n")[0].strip()
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    
    return t.strip()

## 4) Chunking + embeddings + FAISS (index + retrieval)

Cette cellule définit tout le pipeline retrieval.

### `chunk_text_words(text, chunk_words, overlap)`
- Découpe un contexte en chunks (mots) avec chevauchement.

### `mean_pool(...)` + `embed_texts(...)`
- Calcule des embeddings normalisés (cosine via inner product).

### `save_chunks_jsonl` / `load_chunks_jsonl`
- Sauvegarde/charge les chunks (JSONL).

### `build_or_load_faiss_index(ds_train, ...)`
- **Entrées :** split corpus (`ds_train["context"]`), retriever, paramètres chunking.
- **Ce que fait la fonction :**
  - construit la liste de chunks (avec dédup des contexts),
  - embed les chunks,
  - ajoute dans FAISS (IndexFlatIP),
  - sauvegarde index + chunks en cache.
- **Sorties :** `index` FAISS + `corpus_chunks`.

### `retrieve_topk_contexts(questions, ...)`
- **Entrées :** liste de questions, index FAISS + chunks.
- **Ce que fait la fonction :**
  - embed les questions,
  - cherche les top-k chunks dans l’index,
  - concatène les chunks en un seul contexte par question.
- **Sortie :** liste `retrieved_contexts` (1 contexte par question).

In [ ]:
def chunk_text_words(text, chunk_words, overlap):
    text = (text or "").strip()
    if not text:
        return []
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    chunks = []
    step = max(1, chunk_words - overlap)
    for start in range(0, len(words), step):
        end = start + chunk_words
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(words):
            break
    return chunks
    
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom

@torch.inference_mode()
def embed_texts(
    texts,
    tokenizer,
    model,
    max_tokens = 512,
    batch_size = 32,
    prefix = None,
    device = "cuda"):

    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        if prefix is not None:
            batch = [prefix + t for t in batch]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_tokens,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        pooled = F.normalize(pooled, p=2, dim=1)
        vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
        del enc, out, pooled
    return np.vstack(vecs) if vecs else np.zeros((0, 1024), dtype=np.float32)

def save_chunks_jsonl(path, chunks):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for t in chunks:
            f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")

def load_chunks_jsonl(path):
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            chunks.append(obj["text"])
    return chunks


def build_or_load_faiss_index(
    ds_train,
    retr_tok,
    retr_model,
    dim = 1024):
    os.makedirs(CACHE_DIR, exist_ok=True)

    if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
        index = faiss.read_index(INDEX_PATH)
        chunks = load_chunks_jsonl(CHUNKS_PATH)
        return index, chunks

    
    raw_contexts = ds_train["context"]
    seen = set()
    corpus_chunks = []

    for ctx in raw_contexts:
        ctx = (ctx or "").strip()
        if not ctx:
            continue
        
        if ctx in seen:
            continue
        seen.add(ctx)
        parts = chunk_text_words(ctx, CHUNK_WORDS, CHUNK_OVERLAP)
        corpus_chunks.extend(parts)

    
    index = faiss.IndexFlatIP(dim)

    
    for i in tqdm(range(0, len(corpus_chunks), RET_BATCH_SIZE), desc="Indexing corpus (embeddings + FAISS)"):
        batch = corpus_chunks[i:i+RET_BATCH_SIZE]
        emb = embed_texts(
            batch,
            tokenizer=retr_tok,
            model=retr_model,
            max_tokens=RET_MAX_TOKENS,
            batch_size=min(RET_BATCH_SIZE, 64),
            prefix="passage: ",
            device=DEVICE,
        )
        if emb.shape[0] > 0:
            index.add(emb)

    faiss.write_index(index, INDEX_PATH)
    save_chunks_jsonl(CHUNKS_PATH, corpus_chunks)
    return index, corpus_chunks

def retrieve_topk_contexts(
    questions,
    retr_tok,
    retr_model,
    index,
    chunks,
    top_k = 5):
    retrieved_contexts = []
    for i in tqdm(range(0, len(questions), QUERY_BATCH_SIZE), desc="Retrieving top-k contexts"):
        batch_q = questions[i:i+QUERY_BATCH_SIZE]
        q_emb = embed_texts(
            batch_q,
            tokenizer=retr_tok,
            model=retr_model,
            max_tokens=RET_MAX_TOKENS,
            batch_size=min(QUERY_BATCH_SIZE, 64),
            prefix="query: ",
            device=DEVICE,
        )
        D, I = index.search(q_emb, top_k)
        for row in I:
            texts = [chunks[int(idx)] for idx in row if int(idx) >= 0 and int(idx) < len(chunks)]
            
            retrieved_contexts.append("\n\n".join(texts))
    return retrieved_contexts

## 5) Chargement du LLM (7B vs 14B)

Cette cellule définit deux fonctions de chargement du modèle :

- **`load_llm_qwen_7b()`**
  - charge le tokenizer et le modèle Qwen 7B,
  - quantization **4-bit** pour réduire la VRAM,
  - `device_map={"": 0}` (GPU).

- **`load_llm_qwen_14b()`**
  - charge le tokenizer et le modèle 14B (JungZoona),
  - quantization **4-bit**,
  - `device_map="auto"` (dispatch automatique, utile quand la VRAM est limitée).

- **Entrées :** aucune.
- **Sorties :** fonctions prêtes à être appelées.

In [8]:
def load_llm_qwen_7b():
    llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, use_fast=True)
    llm_tok.padding_side = "left"
    llm_tok.truncation_side = "left"
    if llm_tok.pad_token_id is None:
        llm_tok.pad_token = llm_tok.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
    )
    llm.eval()
    llm.config.use_cache = False
    return llm_tok, llm


def load_llm_qwen_14b():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    tok = AutoTokenizer.from_pretrained(LLM_NAME, use_fast=True)
    tok.padding_side = "left"
    tok.truncation_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    llm.eval()
    llm.config.use_cache = False
    return tok, llm

## 6) Chargement des données + construction du retrieval + hit rate

Cette cellule exécute le pipeline retrieval sur la validation.

- **Entrées :**
  - `data/train.csv`, `data/validation.csv`
- **Ce que fait la cellule :**
  1) fixe la graine (`set_seed`),
  2) charge les CSV avec `load_dataset`,
  3) applique `normalize_dataset`,
  4) charge le retriever (`RETRIEVER_NAME`),
  5) construit ou recharge l’index FAISS + chunks depuis le cache,
  6) récupère un contexte top-k pour chaque question de validation,
  7) calcule le **retrieval hit rate** : % des exemples où la réponse gold apparaît dans le contexte récupéré,
  8) libère la VRAM du retriever avant de charger le LLM.
- **Sorties :**
  - `val_questions` : liste des questions validation,
  - `retrieved_ctxs` : liste des contextes récupérés (1 par question),
  - `retrieval_hit_rate` : hit rate en %.

In [9]:
set_seed(SEED)
os.makedirs("results", exist_ok=True)

# Load data
ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})
ds = ds.map(normalize_dataset)

train_split = ds["train"]
val_split = ds["validation"]

Generating train split: 87599 examples [00:00, 168778.14 examples/s]
Generating validation split: 1000 examples [00:00, 128435.07 examples/s]
Map: 100%|████████████████████████| 1000/1000 [00:00<00:00, 29972.16 examples/s]


In [10]:
# Load retriever (embeddings model)
retr_tok = AutoTokenizer.from_pretrained(RETRIEVER_NAME, use_fast=True)
retr_model = AutoModel.from_pretrained(
    RETRIEVER_NAME,
    torch_dtype=torch.float16,
)
retr_model.eval()
retr_model.to(DEVICE)

`torch_dtype` is deprecated! Use `dtype` instead!


Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [11]:
# Build or load FAISS index
index, corpus_chunks = build_or_load_faiss_index(val_split, retr_tok, retr_model, dim=1024)

# Retrieve contexts for validation questions
val_questions = list(val_split["question"])
retrieved_ctxs = retrieve_topk_contexts(val_questions, retr_tok, retr_model, index, corpus_chunks, top_k=TOP_K)


def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

hits = 0
total = len(val_split)
for i in range(total):
    gold = val_split[i]["answers"]["text"][0] if val_split[i]["answers"]["text"] else ""
    if gold and normalize_text(gold) in normalize_text(retrieved_ctxs[i]):
        hits += 1

retrieval_hit_rate = 100.0 * hits / max(1, total)
print(f"Retrieval hit rate (gold in retrieved context): {retrieval_hit_rate:.2f}%")

del retr_model
del retr_tok
torch.cuda.empty_cache()
gc.collect()

Retrieving top-k contexts: 100%|████████████████| 16/16 [00:01<00:00,  8.22it/s]


Retrieval hit rate (gold in retrieved context): 90.20%


419

## 7) Sélection du LLM + génération (RAG)

Cette cellule charge le LLM (7B ou 14B) et génère une réponse pour chaque question validation, en utilisant le contexte récupéré.

- **Entrées :**
  - `LLM_NAME, LLM_TAG` (choix du modèle),
  - `val_questions` + `retrieved_ctxs`,
  - paramètres de génération (`MAX_INPUT_TOKENS`, `MAX_NEW_TOKENS`, `GEN_BATCH_SIZE`).
- **Ce que fait la cellule :**
  1) définit `load_llm()` pour sélectionner 7B ou 14B,
  2) charge le tokenizer + modèle,
  3) boucle sur la validation :
     - construit les prompts (Context + Question),
     - tokenise avec troncature (`MAX_INPUT_TOKENS`),
     - envoie les tensors sur le device du modèle (`llm.device`),
     - génère une réponse courte (greedy),
     - nettoie/stocker sous forme `{"id": ..., "prediction_text": ...}`.
- **Sortie :**
  - `predictions` : liste de prédictions au format SQuAD.

In [ ]:
def load_llm():
    if LLM_TAG == "14B":
        return load_llm_qwen_14b()   
    else:
        return load_llm_qwen_7b()   


try:
    llm_tok, llm = load_llm()
except torch.cuda.OutOfMemoryError:
    if "retr_model" in locals():
        del retr_model
    if "retr_tok" in locals():
        del retr_tok
    torch.cuda.empty_cache()
    gc.collect()
    llm_tok, llm = load_llm()

Loading checkpoint shards: 100%|██████████████████| 4/4 [00:11<00:00,  2.93s/it]


In [ ]:

squad_metric = evaluate.load("squad")
predictions = []
rows_200 = []

ids = list(val_split["id"])
gold_answers = [ex["answers"]["text"][0] if ex["answers"]["text"] else "" for ex in val_split]

for start in tqdm(range(0, len(val_split), GEN_BATCH_SIZE), desc=f"Generating (RAG, Qwen {LLM_TAG})"):
    batch_ids = ids[start:start + GEN_BATCH_SIZE]
    batch_q = val_questions[start:start + GEN_BATCH_SIZE]
    batch_ctx = retrieved_ctxs[start:start + GEN_BATCH_SIZE]

    prompts = [build_prompt(q, llm_tok, context=c) for q, c in zip(batch_q, batch_ctx)]
    inputs = llm_tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}

    with torch.inference_mode():
        out = llm.generate(
            **inputs,
            use_cache=False,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=llm_tok.eos_token_id,
            pad_token_id=llm_tok.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[:, prompt_len:]
    gen_texts = llm_tok.batch_decode(gen_ids, skip_special_tokens=True)

    del inputs, out, gen_ids
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    for ex_id, gt in zip(batch_ids, gen_texts):
        pred = clean_generation(gt)
        predictions.append({"id": str(ex_id), "prediction_text": pred})

Generating (RAG, Qwen 7B): 100%|██████████| 1000/1000 [2:17:01<00:00,  8.22s/it]


## 8) Évaluation (Exact Match / F1) + tableau récapitulatif

Cette cellule calcule les métriques officielles SQuAD.

- **Entrées :**
  - `predictions` (réponses générées),
  - `val_split` (réponses gold dans `answers`),
  - `retrieval_hit_rate`.
- **Ce que fait la cellule :**
  1) construit `references` au format SQuAD,
  2) calcule `exact_match` et `f1`,
  3) construit `metrics_df` (scores + configuration + hit rate),
  4) affiche le tableau.
- **Sorties :**
  - `results` : dict EM/F1,
  - `metrics_df` : tableau des résultats.

In [14]:
references = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in val_split]
results = squad_metric.compute(predictions=predictions, references=references)

metrics_df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
    "model": LLM_NAME,
    "retriever": RETRIEVER_NAME,
    "top_k": TOP_K,
    "chunk_words": CHUNK_WORDS,
    "chunk_overlap": CHUNK_OVERLAP,
    "retrieval_hit_rate": retrieval_hit_rate,
}])

print(f"\n=== RAG (Qwen {LLM_TAG}) Validation Results ===")
print(metrics_df.to_string(index=False))


=== RAG (Qwen 7B) Validation Results ===
     split  exact_match        f1                    model                 retriever  top_k  chunk_words  chunk_overlap  retrieval_hit_rate
validation         43.0 50.684504 Qwen/Qwen2.5-7B-Instruct Qwen/Qwen3-Embedding-0.6B      5          200             50                90.2


## 9) Sauvegarde (métriques + CSV manuel 200)

Cette cellule enregistre les résultats sur disque.

- **Entrées :**
  - `metrics_df`,
  - `predictions`,
  - `val_questions`, `gold_answers`.
- **Ce que fait la cellule :**
  - sauvegarde `metrics_df` dans `METRICS_CSV`,
  - construit un CSV “manuel” sur les 200 premiers :
    - `id`, `question`, `gold_answer`, `generated_answer`,
  - sauvegarde ce CSV dans `PRED200_CSV`.
- **Sorties :**
  - fichiers dans `results/`.

In [ ]:
metrics_df.to_csv(METRICS_CSV, index=False)


pred_by_id = {p["id"]: p["prediction_text"] for p in predictions}
pred200 = []
for i in range(min(200, len(val_split))):
    ex_id = str(ids[i])
    pred200.append({
        "id": ex_id,
        "question": val_questions[i],
        "gold_answer": gold_answers[i],
        "generated_answer": pred_by_id.get(ex_id, ""),
    })
pred200_df = pd.DataFrame(pred200)
pred200_df.to_csv(PRED200_CSV, index=False)

print("\nSaved:")
print(" -", METRICS_CSV)
print(" -", PRED200_CSV)


Saved:
 - results/metrics_table_qwen7B_rag_validation.csv
 - results/manual_eval_qwen7B_rag.csv
